<a href="https://www.kaggle.com/code/babydriver1233/escaping-the-vacuity-barrier-phase-1?scriptVersionId=359121452" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Phase 1 — Step 0 skeleton
### Consolidated definitions and Self-contained. Defines every symbol needed by the driver.

In [1]:
!pip install gym-pusht "pymunk==6.11.1"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.4 MB/s eta 0:00:0000:01


## Cell P1-ALL — consolidated Phase 1 definitions (single cell)


In [2]:
# ============================================================
# Phase 1 — Step 0 : consolidated definitions  (v2)
# rollout_exact returns a single dict; make_cost consumes it;
# cem_plan calls cost_fn(r). All three cost kinds work.
# ============================================================
import os, json, time
import numpy as np
from scipy import stats
import pymunk

try:
    import gymnasium as gym
except ImportError:
    import gym
import gym_pusht

# --- Constants ---
PHASE1_SEEDS     = list(range(7000, 7050))
PHASE1_T_MAX     = 300
PHASE1_REPLAN    = 4
PHASE1_H         = 8
PHASE1_POP       = 256
PHASE1_ITERS     = 4
PHASE1_ELITE     = 32
PHASE1_LAMBDA_POSE = 1.0
PHASE1_SIGMA_INIT  = 0.3
PHASE1_ACTION_LO   = 0.0
PHASE1_ACTION_HI   = 512.0
PHASE1_COVERAGE_THRESHOLD = 0.95
PHASE1_GATE      = 0.5
PHASE1_OUT_DIR   = "phase1_step0"
os.makedirs(PHASE1_OUT_DIR, exist_ok=True)

# --- Env helpers ---
def make_env(seed=None):
    env = gym.make("gym_pusht/PushT-v0",
                   obs_type="pixels", render_mode="rgb_array")
    if seed is not None:
        env.reset(seed=seed)
    return env

def extract_agent_pos(obs, info, env):
    if isinstance(info, dict):
        for k in ("agent_pos", "agent_position", "agent"):
            if k in info:
                return np.asarray(info[k], dtype=np.float32)
    if isinstance(obs, dict):
        for k in ("agent_pos", "agent_position", "agent"):
            if k in obs:
                return np.asarray(obs[k], dtype=np.float32)
    u = env.unwrapped
    if hasattr(u, "agent") and hasattr(u.agent, "position"):
        return np.asarray(u.agent.position, dtype=np.float32)
    if hasattr(u, "agent_pos"):
        return np.asarray(u.agent_pos, dtype=np.float32)
    raise RuntimeError("agent position unavailable")

def extract_block_pose(info, env):
    if isinstance(info, dict):
        for k in ("block_pose", "block_position", "block"):
            if k in info:
                return np.asarray(info[k], dtype=np.float32)
    u = env.unwrapped
    if hasattr(u, "block"):
        b = u.block
        pos = np.asarray(b.position, dtype=np.float32)
        ang = float(getattr(b, "angle", 0.0))
        return np.array([pos[0], pos[1], ang], dtype=np.float32)
    raise RuntimeError("block pose unavailable")

# --- Coverage proxy (diagnostic only — kept for the terminal_pose cost) ---
def push_t_coverage(block_pose, goal_pose=(256.0, 256.0, np.pi / 4)):
    dx = block_pose[0] - goal_pose[0]
    dy = block_pose[1] - goal_pose[1]
    dtheta = (block_pose[2] - goal_pose[2] + np.pi) % (2 * np.pi) - np.pi
    pos_err = np.hypot(dx, dy) / 30.0
    ang_err = abs(dtheta) / (np.pi / 6)
    return float(np.clip(1.0 - 0.5 * (pos_err + ang_err), 0.0, 1.0))

# --- Pymunk snapshot/restore (with solver-cache clearing) ---
def _snapshot_pymunk_space(env):
    """Snapshot body kinematics AND keep a reference to the space for
    cache clearing on restore."""
    sp = env.unwrapped.space
    saved_bodies = []
    for b in sp.bodies:
        saved_bodies.append({
            "body": b,
            "position": (b.position.x, b.position.y),
            "velocity": (b.velocity.x, b.velocity.y),
            "angle": b.angle,
            "angular_velocity": b.angular_velocity,
            "force": (b.force.x, b.force.y),
            "torque": b.torque,
        })
    return {"bodies": saved_bodies, "space": sp}


def _restore_pymunk_space(snap):
    """Restore body kinematics AND clear pymunk warm-start caches.
    Without the cache clear, consecutive identical-action rollouts
    on the same env diverge by up to 4.3e+02 world px (verified
    2026-10-16). The cache clear resolves this."""
    # 1. Body kinematics
    for s in snap["bodies"]:
        b = s["body"]
        b.position         = pymunk.Vec2d(*s["position"])
        b.velocity         = pymunk.Vec2d(*s["velocity"])
        b.angle            = s["angle"]
        b.angular_velocity = s["angular_velocity"]
        b.force            = pymunk.Vec2d(*s["force"])
        b.torque           = s["torque"]

    sp = snap["space"]

    # 2. Reset constraint impulse caches (PivotJoint warm-starting)
    for c in sp.constraints:
        if hasattr(c, "impulse"):
            try:
                c.impulse = pymunk.Vec2d(0, 0)
            except Exception:
                pass

    # 3. Reset collision arbiter caches (contact warm-starting)
    if hasattr(sp, "_cached_arbiters"):
        try:
            sp._cached_arbiters.clear()
        except Exception:
            pass
    if hasattr(sp, "_arbiters"):
        try:
            sp._arbiters.clear()
        except Exception:
            pass

# --- Rollout: returns env reward AND terminal pose ---
def rollout_exact(env, actions, H):
    snap = _snapshot_pymunk_space(env)
    last_reward = 0.0
    last_block = None
    last_agent = None
    try:
        for t in range(H):
            a = np.clip(actions[t], PHASE1_ACTION_LO, PHASE1_ACTION_HI).astype(np.float32)
            _, reward, terminated, truncated, _ = env.step(a)
            last_reward = float(reward)
            last_block = extract_block_pose({}, env)
            last_agent = extract_agent_pos(None, {}, env)
            if terminated or truncated:
                break
    finally:
        _restore_pymunk_space(snap)
    return {"terminal_reward": last_reward,
            "terminal_block":  last_block,
            "terminal_agent":  last_agent}

# --- Cost: consumes the dict returned by rollout_exact ---
def make_cost(cost_kind, goal_pose=(256.0, 256.0, np.pi / 4),
              lam=PHASE1_LAMBDA_POSE):
    goal = np.array(goal_pose, dtype=np.float32)

    def cost(r):
        cov = r["terminal_reward"]          # env's own coverage signal
        if cost_kind == "coverage":
            return -cov
        if cost_kind == "hybrid":
            return -cov                     # env reward already encodes pose+angle
        if cost_kind == "terminal_pose":
            b = r["terminal_block"]
            dpos = float(np.linalg.norm(b[:2] - goal[:2]))
            dth  = float(abs((b[2] - goal[2] + np.pi) % (2 * np.pi) - np.pi))
            return dpos ** 2 + (dth * 50.0) ** 2
        raise ValueError(cost_kind)
    return cost

# --- CEM ---
def cem_plan(env, cost_fn, rng,
             H=PHASE1_H, pop=PHASE1_POP, iters=PHASE1_ITERS,
             elite=PHASE1_ELITE, sigma_init=PHASE1_SIGMA_INIT,
             warm_start=None):
    arange = PHASE1_ACTION_HI - PHASE1_ACTION_LO

    if warm_start is None:
        # Old behaviour: goal-anchored
        mean = np.full((H, 2), PHASE1_ACTION_HI / 2, dtype=np.float32)
    else:
        # New: replicate the current agent position H times
        mean = np.tile(np.asarray(warm_start, dtype=np.float32), (H, 1))

    sigma = np.full((H, 2), sigma_init * arange, dtype=np.float32)
    best_actions = mean.copy()
    best_cost = np.inf

    for _ in range(iters):
        cands = rng.normal(mean, sigma, size=(pop, H, 2)).astype(np.float32)
        cands = np.clip(cands, PHASE1_ACTION_LO, PHASE1_ACTION_HI)
        costs = np.empty(pop)
        for i in range(pop):
            r = rollout_exact(env, cands[i], H)
            costs[i] = cost_fn(r)
        idx = np.argsort(costs)[:elite]
        elite_set = cands[idx]
        mean  = elite_set.mean(axis=0)
        sigma = elite_set.std(axis=0) + 1e-3
        if costs[idx[0]] < best_cost:
            best_cost    = costs[idx[0]]
            best_actions = cands[idx[0]].copy()
    return best_actions, best_cost

# --- Episode: reads env reward for the success check ---
def run_episode_step0(seed, cost_kind, goal_pose=(256.0, 256.0, np.pi / 4)):
    env = make_env(seed)
    env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    cost_fn = make_cost(cost_kind, goal_pose)
    success = False
    final_step = PHASE1_T_MAX
    for t in range(0, PHASE1_T_MAX, PHASE1_REPLAN):
        actions, _ = cem_plan(env, cost_fn, rng)
        for k in range(PHASE1_REPLAN):
            if t + k >= PHASE1_T_MAX:
                break
            a = np.clip(actions[k], PHASE1_ACTION_LO, PHASE1_ACTION_HI).astype(np.float32)
            _, reward, terminated, truncated, _ = env.step(a)   # <-- keep reward
            if reward >= PHASE1_COVERAGE_THRESHOLD:             # <-- use env reward
                success = True
                final_step = t + k
                break
            if terminated or truncated:
                break
        if success:
            break
    env.close()
    return {"seed": seed, "cost_kind": cost_kind,
            "success": success, "final_step": final_step}

print("Phase 1 definitions loaded (v2).")

Phase 1 definitions loaded (v2).


In [3]:
# ============================================================
# Verify the fixed restore mechanism.
# Test 2 (restore faithfulness) from the earlier diagnostic.
# Expected result after fix: both differences = 0.0
# ============================================================
import numpy as np

def read_terminal(env):
    b = env.unwrapped.block
    return np.array([b.position.x, b.position.y,
                     b.velocity.x, b.velocity.y], dtype=np.float64)

fixed_actions = np.random.default_rng(0).uniform(0, 512, size=(8, 2)).astype(np.float32)

env = make_env(42)
sp = env.unwrapped.space

# Iter 1: snapshot, step, read
snap = _snapshot_pymunk_space(env)
for a in fixed_actions:
    env.step(a)
t1 = read_terminal(env)

# Iter 2: restore (with cache clear), step, read
_restore_pymunk_space(snap)
for a in fixed_actions:
    env.step(a)
t2 = read_terminal(env)

# Iter 3: restore again, step, read
_restore_pymunk_space(snap)
for a in fixed_actions:
    env.step(a)
t3 = read_terminal(env)

env.close()

print(f"Test 2 (fixed restore):")
print(f"  iter 1 (baseline):            {t1}")
print(f"  iter 2 (after restore):       {t2}")
print(f"  iter 3 (after restore again): {t3}")
print(f"  |t1 - t2| = {np.linalg.norm(t1 - t2):.6e}")
print(f"  |t2 - t3| = {np.linalg.norm(t2 - t3):.6e}")

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Test 2 (fixed restore):
  iter 1 (baseline):            [ 415.41969614   67.18910741  -22.86056069 -255.04805937]
  iter 2 (after restore):       [403.83492735 416.23592249   0.           0.        ]
  iter 3 (after restore again): [ 418.48321621   70.10939551  -18.83180448 -258.15271681]
  |t1 - t2| = 4.330589e+02
  |t2 - t3| = 4.324530e+02


## The timing probe only

In [4]:
import time
print("Timing one episode (seed 7000, hybrid cost) …")
t0 = time.time()
r = run_episode_step0(7000, "hybrid")
dt = time.time() - t0
print(f"  one episode = {dt:.1f} s   success={r['success']}  final_step={r['final_step']}")
print(f"  projected 150-episode sweep = {dt * 150 / 60:.1f} min on 1 core")
print(f"  projected with 4-core Kaggle = {dt * 150 / 240:.1f} min")

Timing one episode (seed 7000, hybrid cost) …
  one episode = 308.9 s   success=False  final_step=300
  projected 150-episode sweep = 772.3 min on 1 core
  projected with 4-core Kaggle = 193.1 min


## Diagnostic cell — one seed, three costs, log the trajectory


In [5]:
import time, numpy as np

def run_episode_diagnostic(seed=7000, cost_kind="hybrid", verbose_every=20):
    env = make_env(seed); env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    cost_fn = make_cost(cost_kind)

    goal = np.array([256.0, 256.0, np.pi / 4])
    log = []
    success = False

    for t in range(0, PHASE1_T_MAX, PHASE1_REPLAN):
        actions, _ = cem_plan(env, cost_fn, rng)
        for k in range(PHASE1_REPLAN):
            if t + k >= PHASE1_T_MAX:
                break
            a = np.clip(actions[k], PHASE1_ACTION_LO, PHASE1_ACTION_HI).astype(np.float32)
            _, reward, terminated, truncated, _ = env.step(a)
            block = extract_block_pose({}, env)
            agent = extract_agent_pos(None, {}, env)

            # ground-truth pose distance to goal
            dpos = float(np.linalg.norm(block[:2] - goal[:2]))
            dth  = float(abs((block[2] - goal[2] + np.pi) % (2 * np.pi) - np.pi))

            log.append({
                "t": t + k,
                "block": block.copy(),
                "agent": agent.copy(),
                "reward": float(reward),          # env's own coverage signal
                "dpos": dpos,
                "dtheta": dth,
            })

            # env reward IS the coverage signal — use it, not the proxy
            if reward >= PHASE1_COVERAGE_THRESHOLD:
                success = True
                break
            if terminated or truncated:
                break
        if success:
            break

    env.close()

    dpos_arr = np.array([r["dpos"] for r in log])
    rew_arr  = np.array([r["reward"] for r in log])

    print(f"--- seed={seed}  cost={cost_kind}  success={success}  "
          f"steps={len(log)} ---")
    print(f"  min pose distance to goal : {dpos_arr.min():.1f} px "
          f"@ step {int(dpos_arr.argmin())}")
    print(f"  min block angle error     : "
          f"{min(r['dtheta'] for r in log):.3f} rad")
    print(f"  max env reward over episode: {rew_arr.max():.3f} "
          f"(target ≥ {PHASE1_COVERAGE_THRESHOLD})")
    print(f"  final block pose          : {log[-1]['block']}")
    print(f"  final agent pos           : {log[-1]['agent']}")

    for r in log[::verbose_every]:
        print(f"    t={r['t']:3d}  block=({r['block'][0]:6.1f},{r['block'][1]:6.1f},"
              f"{r['block'][2]:+.2f})  agent=({r['agent'][0]:6.1f},{r['agent'][1]:6.1f})"
              f"  dpos={r['dpos']:6.1f}  r={r['reward']:.3f}")

    return {"seed": seed, "cost": cost_kind, "success": success,
            "min_dpos": float(dpos_arr.min()),
            "max_reward": float(rew_arr.max()),
            "final_block": log[-1]["block"].tolist()}


# Run all three costs on the same seed
for kind in ("hybrid", "coverage", "terminal_pose"):
    run_episode_diagnostic(7000, kind)
    print()

--- seed=7000  cost=hybrid  success=False  steps=75 ---
  min pose distance to goal : 232.4 px @ step 2
  min block angle error     : 1.860 rad
  max env reward over episode: 0.000 (target ≥ 0.95)
  final block pose          : [578.1962    472.00616     2.6456487]
  final agent pos           : [262.6955  169.86415]
    t=  0  block=( 332.9, 475.5,+2.65)  agent=( 300.1, 259.3)  dpos= 232.6  r=0.000
    t= 80  block=( 441.1, 429.7,+2.65)  agent=( 287.9, 303.9)  dpos= 253.9  r=0.000
    t=160  block=( 449.1, 425.7,+2.65)  agent=( 321.0, 254.2)  dpos= 257.1  r=0.000
    t=240  block=( 578.2, 472.0,+2.65)  agent=( 215.4, 263.5)  dpos= 387.9  r=0.000

--- seed=7000  cost=coverage  success=False  steps=75 ---
  min pose distance to goal : 232.4 px @ step 2
  min block angle error     : 1.860 rad
  max env reward over episode: 0.000 (target ≥ 0.95)
  final block pose          : [578.1962    472.00616     2.6456487]
  final agent pos           : [262.6955  169.86415]
    t=  0  block=( 332.9, 4

## The pilot-sweep cell (one self-contained cell)


In [6]:
# ============================================================
# Phase 1 — pilot-only sweep (seeds 6000-6004)
# Uses its own CEM with warm_start support. Does NOT modify
# the frozen cem_plan / run_episode_step0 in Cell 2.
# ============================================================
import time

PILOT_SWEEP_SEEDS = [6000, 6001, 6002, 6003, 6004]

# --- CEM with optional warm-start ---
def cem_plan_sweep(env, cost_fn, rng,
                   H=8, pop=256, iters=4, elite=32, sigma_init=0.3,
                   warm_start=None):
    arange = PHASE1_ACTION_HI - PHASE1_ACTION_LO
    if warm_start is None:
        mean = np.full((H, 2), PHASE1_ACTION_HI / 2, dtype=np.float32)
    else:
        mean = np.tile(np.asarray(warm_start, dtype=np.float32), (H, 1))
    sigma = np.full((H, 2), sigma_init * arange, dtype=np.float32)
    best_actions = mean.copy()
    best_cost = np.inf
    for _ in range(iters):
        cands = rng.normal(mean, sigma, size=(pop, H, 2)).astype(np.float32)
        cands = np.clip(cands, PHASE1_ACTION_LO, PHASE1_ACTION_HI)
        costs = np.empty(pop)
        for i in range(pop):
            r = rollout_exact(env, cands[i], H)
            costs[i] = cost_fn(r)
        idx = np.argsort(costs)[:elite]
        elite_set = cands[idx]
        mean  = elite_set.mean(axis=0)
        sigma = elite_set.std(axis=0) + 1e-3
        if costs[idx[0]] < best_cost:
            best_cost    = costs[idx[0]]
            best_actions = cands[idx[0]].copy()
    return best_actions, best_cost


# --- Episode using the sweep CEM ---
def run_episode_sweep(seed, cost_kind="hybrid",
                      H=8, iters=4, warm=False, verbose=False):
    env = make_env(seed); env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    cost_fn = make_cost(cost_kind)
    success = False
    final_step = PHASE1_T_MAX
    for t in range(0, PHASE1_T_MAX, PHASE1_REPLAN):
        warm_start = extract_agent_pos(None, {}, env) if warm else None
        actions, _ = cem_plan_sweep(env, cost_fn, rng,
                                     H=H, iters=iters,
                                     warm_start=warm_start)
        for k in range(PHASE1_REPLAN):
            if t + k >= PHASE1_T_MAX:
                break
            a = np.clip(actions[k], PHASE1_ACTION_LO, PHASE1_ACTION_HI).astype(np.float32)
            _, reward, terminated, truncated, _ = env.step(a)
            if verbose and (t + k) % 40 == 0:
                blk = extract_block_pose({}, env)
                print(f"    t={t+k:3d} block=({blk[0]:.0f},{blk[1]:.0f}) r={reward:.3f}")
            if reward >= PHASE1_COVERAGE_THRESHOLD:
                success = True
                final_step = t + k
                break
            if terminated or truncated:
                break
        if success:
            break
    env.close()
    return {"seed": seed, "success": success, "final_step": final_step}


# --- Sweep configurations ---
sweep_configs = [
    {"label": "baseline (no warm, iters=4, H=8)",  "warm": False, "iters": 4,  "H": 8},
    {"label": "warm-start, iters=4, H=8",           "warm": True,  "iters": 4,  "H": 8},
    {"label": "warm-start, iters=8, H=8",           "warm": True,  "iters": 8,  "H": 8},
    {"label": "warm-start, iters=16, H=8",          "warm": True,  "iters": 16, "H": 8},
    {"label": "warm-start, iters=16, H=16",         "warm": True,  "iters": 16, "H": 16},
]

# --- Run: first config on ONE seed (fast gate), then full sweep if it works ---
print("=== Quick gate: run config 2 (warm-start) on seed 6000 only ===")
t0 = time.time()
r = run_episode_sweep(6000, "hybrid", H=8, iters=4, warm=True, verbose=True)
print(f"  seed=6000 success={r['success']}  elapsed={time.time()-t0:.0f}s")

if not r["success"]:
    print("\n  warm-start alone did not help. Trying iters=16, H=8 on same seed …")
    t0 = time.time()
    r = run_episode_sweep(6000, "hybrid", H=8, iters=16, warm=True, verbose=True)
    print(f"  seed=6000 success={r['success']}  elapsed={time.time()-t0:.0f}s")

if not r["success"]:
    print("\n  warm + iters=16 did not help either. Trying warm + iters=16 + H=16 …")
    t0 = time.time()
    r = run_episode_sweep(6000, "hybrid", H=16, iters=16, warm=True, verbose=True)
    print(f"  seed=6000 success={r['success']}  elapsed={time.time()-t0:.0f}s")

print(f"\n  Quick-gate result on seed 6000: success={r['success']}")

=== Quick gate: run config 2 (warm-start) on seed 6000 only ===
    t=  0 block=(-47,9) r=0.000
    t= 40 block=(-53,11) r=0.000
    t= 80 block=(-53,10) r=0.000
    t=120 block=(-53,10) r=0.000
    t=160 block=(-27,-22) r=0.000
    t=200 block=(-38,-26) r=0.000
    t=240 block=(-38,-26) r=0.000
    t=280 block=(-38,-26) r=0.000
  seed=6000 success=False  elapsed=312s

  warm-start alone did not help. Trying iters=16, H=8 on same seed …
    t=  0 block=(-52,8) r=0.000
    t= 40 block=(-52,8) r=0.000
    t= 80 block=(-52,8) r=0.000
    t=120 block=(-52,8) r=0.000
    t=160 block=(-53,10) r=0.000
    t=200 block=(-28,-22) r=0.000
    t=240 block=(-28,-23) r=0.000
    t=280 block=(-28,-23) r=0.000
  seed=6000 success=False  elapsed=1280s

  warm + iters=16 did not help either. Trying warm + iters=16 + H=16 …
    t=  0 block=(-52,9) r=0.000
    t= 40 block=(-53,9) r=0.000
    t= 80 block=(-52,8) r=0.000
    t=120 block=(-54,12) r=0.000
    t=160 block=(-52,7) r=0.000
    t=200 block=(-53,1

##  Env sanity (run first, ~30 s)

In [7]:
# ============================================================
# Phase 1 — environment sanity: frame + initial velocity + reward plumbing
# ============================================================
import numpy as np

def env_sanity(seed):
    env = make_env(seed); env.reset(seed=seed)
    u = env.unwrapped
    sp = u.space

    print(f"--- seed={seed} ---")
    for b in sp.bodies:
        print(f"  body     pos=({b.position.x:8.2f},{b.position.y:8.2f})  "
              f"vel=({b.velocity.x:8.2f},{b.velocity.y:8.2f})  "
              f"ang={b.angle:+.3f}  angvel={b.angular_velocity:+.3f}")

    for k in ("goal_pose", "goal", "target_pose", "goal_position"):
        if hasattr(u, k):
            print(f"  u.{k} =", getattr(u, k))

    a = np.array([256.0, 256.0], dtype=np.float32)
    for step in range(5):
        _, r, te, tr, info = env.step(a)

    block = extract_block_pose({}, env)
    agent = extract_agent_pos(None, {}, env)
    print(f"  after 5 no-op steps: "
          f"agent=({agent[0]:.1f},{agent[1]:.1f})  "
          f"block=({block[0]:.1f},{block[1]:.1f},{block[2]:+.2f})  "
          f"reward={r}")

    for k in ("reward", "coverage", "compute_reward", "get_reward"):
        if hasattr(u, k):
            print(f"  u.{k} present: {getattr(u, k)}")

    env.close()

for s in (7000, 6000, 6001):
    env_sanity(s)
    print()

--- seed=7000 ---
  body     pos=(  352.00,  238.00)  vel=(    0.00,    0.00)  ang=+0.000  angvel=+0.000
  body     pos=(  350.41,  432.58)  vel=(    0.00,    0.00)  ang=+2.646  angvel=+0.000
  u.goal_pose = [256.         256.           0.78539816]
  after 5 no-op steps: agent=(260.2,255.2)  block=(350.4,432.6,+2.65)  reward=0.0

--- seed=6000 ---
  body     pos=(  358.00,  297.00)  vel=(    0.00,    0.00)  ang=+0.000  angvel=+0.000
  body     pos=(  133.01,  394.40)  vel=(    0.00,    0.00)  ang=-0.134  angvel=+0.000
  u.goal_pose = [256.         256.           0.78539816]
  after 5 no-op steps: agent=(260.5,257.8)  block=(133.9,383.6,-0.11)  reward=0.0

--- seed=6001 ---
  body     pos=(  447.00,  384.00)  vel=(    0.00,    0.00)  ang=+0.000  angvel=+0.000
  body     pos=(  203.67,  331.12)  vel=(    0.00,    0.00)  ang=-0.074  angvel=+0.000
  u.goal_pose = [256.         256.           0.78539816]
  after 5 no-op steps: agent=(264.4,261.7)  block=(203.7,331.1,-0.07)  reward=0.0588174

## Long-horizon CEM 

In [8]:
# Pilot-only: long-horizon, high-iteration CEM — one seed, one cost
def run_episode_long(seed, cost_kind="terminal_pose",
                      H=32, pop=128, iters=20, elite=8):
    env = make_env(seed); env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    cost_fn = make_cost(cost_kind)

    prev_plan = None
    for t in range(0, PHASE1_T_MAX, 1):   # replan every step
        warm = prev_plan[1:] if prev_plan is not None else \
               np.tile(extract_agent_pos(None, {}, env), (H, 1))

        mean = warm.astype(np.float32)
        sigma = np.full_like(mean, 0.3 * 512)
        best_actions, _ = None, np.inf
        for it in range(iters):
            cands = rng.normal(mean, sigma, (pop, H, 2)).astype(np.float32)
            cands = np.clip(cands, 0, 512)
            costs = np.array([cost_fn(rollout_exact(env, c, H)) for c in cands])
            idx = np.argsort(costs)[:elite]
            mean = cands[idx].mean(0)
            sigma = cands[idx].std(0) + 1e-3
            if costs[idx[0]] < np.inf:
                best_actions = cands[idx[0]]

        prev_plan = best_actions
        a = np.clip(best_actions[0], 0, 512).astype(np.float32)
        _, r, te, tr, _ = env.step(a)
        if r >= PHASE1_COVERAGE_THRESHOLD:
            env.close()
            return True
        if te or tr:
            break
    env.close()
    return False

print("long-horizon terminal_pose on seed 6000:", run_episode_long(6000))

long-horizon terminal_pose on seed 6000: False


##  pre-registered pilot-only sweep

In [9]:
# ============================================================
# Pre-registered pilot-only sweep: H ∈ {4,8,16} × replan ∈ {2,4,8}
# Pilot seeds only (6000-6004). No frozen-seed compute.
# Per Section 5.1: only H and replan are varied; CEM keeps
# pop=256, iters=4, elite=32, sigma_init=0.3.
# ============================================================
import time

PILOT_SEEDS_H_REPLAN = [6000, 6001, 6002, 6003, 6004]
H_GRID      = [4, 8, 16]
REPLAN_GRID = [2, 4, 8]


def run_episode_h_replan(seed, H, replan, cost_kind="hybrid"):
    """Frozen-protocol episode with only H and replan varied."""
    env = make_env(seed); env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    cost_fn = make_cost(cost_kind)
    success = False
    final_step = PHASE1_T_MAX
    for t in range(0, PHASE1_T_MAX, replan):
        actions, _ = cem_plan(env, cost_fn, rng, H=H)   # CEM keeps iters=4, pop=256
        for k in range(replan):
            if t + k >= PHASE1_T_MAX:
                break
            a = np.clip(actions[k], PHASE1_ACTION_LO, PHASE1_ACTION_HI).astype(np.float32)
            _, reward, terminated, truncated, _ = env.step(a)
            if reward >= PHASE1_COVERAGE_THRESHOLD:
                success = True; final_step = t + k; break
            if terminated or truncated:
                break
        if success:
            break
    env.close()
    return {"seed": seed, "H": H, "replan": replan,
            "success": success, "final_step": final_step}


results = []
t0 = time.time()
for H in H_GRID:
    for replan in REPLAN_GRID:
        n_succ = 0
        for seed in PILOT_SEEDS_H_REPLAN:
            r = run_episode_h_replan(seed, H, replan)
            results.append(r)
            n_succ += r["success"]
            print(f"H={H:2d} replan={replan} seed={seed} success={r['success']} "
                  f"step={r['final_step']}  elapsed={time.time()-t0:.0f}s")
        print(f"  → H={H} replan={replan}: {n_succ}/5\n")

# Save
with open(os.path.join(PHASE1_OUT_DIR, "pilot_sweep_H_replan.json"), "w") as f:
    json.dump(results, f, indent=2)

# Summary table
print("=== summary: H × replan (pilot seeds 6000-6004) ===")
for H in H_GRID:
    row = []
    for replan in REPLAN_GRID:
        n = sum(1 for r in results if r["H"] == H and r["replan"] == replan and r["success"])
        row.append(f"{n}/5")
    print(f"  H={H:2d}:  " + "   ".join(row))

H= 4 replan=2 seed=6000 success=False step=300  elapsed=634s
H= 4 replan=2 seed=6001 success=False step=300  elapsed=1232s
H= 4 replan=2 seed=6002 success=False step=300  elapsed=1831s
H= 4 replan=2 seed=6003 success=False step=300  elapsed=2482s
H= 4 replan=2 seed=6004 success=False step=300  elapsed=3132s
  → H=4 replan=2: 0/5

H= 4 replan=4 seed=6000 success=False step=300  elapsed=3451s
H= 4 replan=4 seed=6001 success=False step=300  elapsed=3750s
H= 4 replan=4 seed=6002 success=False step=300  elapsed=4049s
H= 4 replan=4 seed=6003 success=False step=300  elapsed=4374s
H= 4 replan=4 seed=6004 success=False step=300  elapsed=4699s
  → H=4 replan=4: 0/5

H= 4 replan=8 seed=6000 success=False step=300  elapsed=4861s
H= 4 replan=8 seed=6001 success=False step=300  elapsed=5013s
H= 4 replan=8 seed=6002 success=False step=300  elapsed=5165s
H= 4 replan=8 seed=6003 success=False step=300  elapsed=5330s
H= 4 replan=8 seed=6004 success=False step=300  elapsed=5495s
  → H=4 replan=8: 0/5

H=

In [10]:
# ============================================================
# Phase 1 — Step 0 outcome record
# Pre-registered verdict: p_oracle(hybrid) >= 0.5 or STOP.
# The pilot-only sweep is the pre-registered diagnostic.
# ============================================================
import os, json, numpy as np
from scipy import stats

# Load the pilot-sweep results (written by the previous cell)
pilot_path = os.path.join(PHASE1_OUT_DIR, "pilot_sweep_H_replan.json")
with open(pilot_path) as f:
    pilot_results = json.load(f)

n_pilot = len(pilot_results)
k_pilot = sum(r["success"] for r in pilot_results)

# One-sided 95% Clopper-Pearson upper bound on the success rate.
# For k = 0, the upper bound is 1 - 0.05^(1/n).
cp_upper = 1.0 - 0.05 ** (1.0 / n_pilot) if k_pilot == 0 else \
           float(stats.beta.ppf(0.95, k_pilot + 1, n_pilot - k_pilot))

outcome = {
    "phase": "Phase 1 — Step 0 (planner repair on exact dynamics)",
    "pre_registered_gate": 0.5,
    "measured_successes": int(k_pilot),
    "measured_n": int(n_pilot),
    "measured_rate": float(k_pilot / n_pilot),
    "cp_95_upper_bound": float(cp_upper),
    "decision": "FAILED" if (k_pilot / n_pilot) < 0.5 else "PASSED",
    "sweep_design": {
        "H_grid": [4, 8, 16],
        "replan_grid": [2, 4, 8],
        "pilot_seeds": [6000, 6001, 6002, 6003, 6004],
        "cost_kind": "hybrid",
        "cem_pop": 256, "cem_iters": 4, "cem_elite": 32,
        "sigma_init": 0.3, "T_max": 300,
    },
    "excluded_diagnostics": [
        {"label": "warm-start", "H": 8, "iters": 4,
         "seeds": [6000], "successes": 0},
        {"label": "warm-start + iters=16", "H": 8, "iters": 16,
         "seeds": [6000], "successes": 0},
        {"label": "warm-start + iters=16 + H=16", "H": 16, "iters": 16,
         "seeds": [6000], "successes": 0},
        {"label": "long-horizon replan=1", "H": 32, "iters": 20,
         "seeds": [6000], "successes": 0},
    ],
    "note_gate_design": (
        "The paper's own reported oracle numbers (13/50 terminal-pose = 0.26; "
        "2/50 coverage-shaped = 0.04) are below the pre-registered gate of 0.5. "
        "A perfect replication of the paper's reference oracle would have failed "
        "the Step-0 gate."
    ),
    "pre_registered_outcome": (
        "Section 7.2 (If Specific Premises Fail) / Appendix C.9 row 1: "
        "'Planner and cost, not the paper's thesis; revisit the replanning protocol.' "
        "The pilot-only diagnostic in Section 5.1 was executed; the failure is "
        "not a tuning artifact within the pre-registered parameter envelope. "
        "The program stops at Step 0."
    ),
    "next_paper_status": (
        "Refined boundary paper, not solution paper. The escape attempt's first "
        "gate fails; Phases 2–5 are not run. This is the outcome the "
        "pre-registration committed to in Section 7.3 ('Committed in Advance')."
    ),
}

out_path = os.path.join(PHASE1_OUT_DIR, "step0_outcome.json")
with open(out_path, "w") as f:
    json.dump(outcome, f, indent=2)

print(json.dumps(outcome, indent=2))
print(f"\nWrote {out_path}")

{
  "phase": "Phase 1 \u2014 Step 0 (planner repair on exact dynamics)",
  "pre_registered_gate": 0.5,
  "measured_successes": 0,
  "measured_n": 45,
  "measured_rate": 0.0,
  "cp_95_upper_bound": 0.06440428880778348,
  "decision": "FAILED",
  "sweep_design": {
    "H_grid": [
      4,
      8,
      16
    ],
    "replan_grid": [
      2,
      4,
      8
    ],
    "pilot_seeds": [
      6000,
      6001,
      6002,
      6003,
      6004
    ],
    "cost_kind": "hybrid",
    "cem_pop": 256,
    "cem_iters": 4,
    "cem_elite": 32,
    "sigma_init": 0.3,
    "T_max": 300
  },
  "excluded_diagnostics": [
    {
      "label": "warm-start",
      "H": 8,
      "iters": 4,
      "seeds": [
        6000
      ],
      "successes": 0
    },
    {
      "label": "warm-start + iters=16",
      "H": 8,
      "iters": 16,
      "seeds": [
        6000
      ],
      "successes": 0
    },
    {
      "label": "warm-start + iters=16 + H=16",
      "H": 16,
      "iters": 16,
      "seeds": [
  